# Hybrid Recommender (Item-Item + Content)

In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
users_movies = pd.read_csv("users_movies.csv")

In [4]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

##  Create User Iem matrix and Item Matrix, find similarity scores for each movie

In [6]:
def sparse_matrix(df):
    movies = df['movie_id'].nunique()
    users = df['user_id'].nunique()

    x = list(range(movies))
    y = list(range(users))
    a = df["movie_id"]
    b = df["user_id"]
    
    m = dict(zip(np.unique(a), x))
    u = dict(zip(np.unique(b), y))
    m_inv = dict(zip(x, np.unique(a)))
    u_inv = dict(zip(y, np.unique(b)))

    m_idx = [m[i] for i in a]
    u_idx = [u[i] for i in b]
    matrix = csr_matrix((df["rating"], (u_idx,m_idx)), shape=(users,movies))
    return matrix, u, m, u_inv, m_inv

matrix, u, m, u_inv, m_inv = sparse_matrix(train)


In [8]:
item_matrix = matrix.T
kNN = NearestNeighbors(n_neighbors=20, algorithm="brute", metric="cosine")
kNN.fit(item_matrix)
neighbors = {}

In [10]:
for movie_id, index in m.items():
    movie_vector = item_matrix[index].reshape(1, -1)
    d, indexs = kNN.kneighbors(movie_vector)

    n = []
    for i in range(1, len(indexs[0])):
        n_index = indexs[0][i]
        if n_index not in m_inv:
            continue
        n_id = m_inv[n_index]
        sim = 1 - d[0][i]
        n.append((n_id, sim))

    neighbors[movie_id] = n


In [12]:
seen = train.groupby("user_id")["movie_id"].apply(set).to_dict()
liked = (train[train["rating"] >= 4].groupby("user_id")["movie_id"].apply(list).to_dict())
test_set = (test[test["rating"] >= 4].groupby("user_id")["movie_id"].apply(list).to_dict())

## Create Genre matrix 

In [14]:
genres = set()
train['genres'] = train['genres'].str.replace('|', ' ')
for i in train['genres']:
    list_i = i.split()
    for j in list_i:
        genres.add(j)
print(genres)

{'Crime', 'Action', 'Documentary', 'Comedy', 'Romance', 'Western', 'Musical', "Children's", 'Horror', 'Fantasy', 'Sci-Fi', 'Mystery', 'Adventure', 'Thriller', 'Film-Noir', 'War', 'Animation', 'Drama'}


In [16]:
movies = train[['movie_id', 'title', 'genres']].drop_duplicates(subset='movie_id')

In [18]:
for i in genres:
    movies[i] = movies.genres.apply(lambda x: int(i in x))
movie_genres = movies.set_index('movie_id').drop(columns=[ 'title', 'genres'])

watched = train.groupby('user_id')['movie_id']
watched = watched.apply(set).to_dict()

## Content based recommender

In [20]:
def content(user_id):
    s = seen.get(user_id, [])
    l  = liked.get(user_id, [])
    l_m = []
    
    for i in l:
        if i in movie_genres.index:
            l_m.append(i)
    if len(l_m) == 0:
        return {}
        
    vex = movie_genres.loc[l_m]
    profile = vex.mean(axis=0)
    profile = profile.values.reshape(1, -1)
    vals = movie_genres.values
    sr = cosine_similarity(profile, vals)[0]

    ids = movie_genres.index.tolist()
    m_sr = dict(zip(ids, sr))
    w_m = watched.get(user, set())
    
    for i in w_m:
        m_sr.pop(i, None)
    return m_sr

## Collaborative recommender

In [22]:
def collab(user):
    l_ids = liked[user]
    s_ids = seen.get(user, set())
    if len(l_ids) == 0:
        return {}
    scores = {}
    for i in l_ids:
        if i not in neighbors:
            continue
        for n, sim in neighbors[i]:
            scores[n] = scores.get(n, 0) + sim * matrix[u[user], m[i]]
    for s in s_ids:
        scores.pop(s, None)
    return scores

## Function to normalize scores

In [24]:
def norm(scores, movies):
    vals = []
    for i in movies:
        vals.append(scores[i])
    max_v = max(vals)
    min_v = min(vals)
    norm_vals = {}
    for i in movies:
        if max_v == min_v:
            norm_vals[i] = 0
        else:
            norm_vals[i] = (scores[i] - min_v)/(max_v - min_v)
    return norm_vals

In [26]:
def ndcg_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    relevant = set(relevant)
    dcg = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0


## Combine both recommenders into a single score for each user in test for top 10 and top 100 recommendations

In [28]:
cf_scores = {}
cb_scores = {}

total_hits = 0
total_relevant = 0
precision_sum = 0
recall_sum = 0
ndcg_sum = 0
num_users = 0

for user, movies in test_set.items():
    if user not in liked:
        continue

    content_scores = content(user)
    collab_scores = collab(user)

    candidate_movies = {i: content_scores[i] for i in content_scores if i in collab_scores}
    if candidate_movies == {}:
        continue

    ct_norm = norm(content_scores, candidate_movies)
    cb_norm = norm(collab_scores, candidate_movies)

    hybrid = {}
    for i in candidate_movies:
        hybrid[i] = (0.7 * cb_norm[i]) + (0.3 * ct_norm[i])

    top_10 = sorted(hybrid, key=hybrid.get, reverse=True)[:10]

    relevant_movies = set(movies)
    hits = len(relevant_movies.intersection(top_10))

    precision = hits / 10
    recall = hits / len(relevant_movies) if len(relevant_movies) > 0 else 0
    ndcg = ndcg_at_k(top_10, relevant_movies, k=10)

    total_hits += hits
    total_relevant += len(relevant_movies)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    num_users += 1

avg_precision = precision_sum / num_users if num_users > 0 else 0
avg_recall = recall_sum / num_users if num_users > 0 else 0
avg_ndcg = ndcg_sum / num_users if num_users > 0 else 0
micro_recall = total_hits / total_relevant if total_relevant > 0 else 0

print("Users evaluated:", num_users)
print("Hits:", total_hits)
print("Possible hits:", total_relevant)
print(f"Precision@10: {avg_precision:.4f}")
print(f"Recall@10: {avg_recall:.4f}")
print(f"NDCG@10: {avg_ndcg:.4f}")
print(f"Micro Recall@10: {micro_recall:.4f}")

Users evaluated: 6015
Hits: 15186
Possible hits: 114951
Precision@10: 0.2525
Recall@10: 0.1728
NDCG@10: 0.3093
Micro Recall@10: 0.1321


In [40]:
cf_scores = {}
cb_scores = {}

total_hits = 0
total_relevant = 0
precision_sum = 0
recall_sum = 0
ndcg_sum = 0
num_users = 0

for user, movies in test_set.items():
    if user not in liked:
        continue

    content_scores = content(user)
    collab_scores = collab(user)

    candidate_movies = {i: content_scores[i] for i in content_scores if i in collab_scores}
    if candidate_movies == {}:
        continue

    ct_norm = norm(content_scores, candidate_movies)
    cb_norm = norm(collab_scores, candidate_movies)

    hybrid = {}
    for i in candidate_movies:
        hybrid[i] = (0.7 * cb_norm[i]) + (0.3 * ct_norm[i])

    top_100 = sorted(hybrid, key=hybrid.get, reverse=True)[:100]

    relevant_movies = set(movies)
    hits = len(relevant_movies.intersection(top_100))

    precision = hits / 100
    recall = hits / len(relevant_movies) if len(relevant_movies) > 0 else 0
    ndcg = ndcg_at_k(top_100, relevant_movies, k=100)

    total_hits += hits
    total_relevant += len(relevant_movies)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    num_users += 1

avg_precision = precision_sum / num_users if num_users > 0 else 0
avg_recall = recall_sum / num_users if num_users > 0 else 0
avg_ndcg = ndcg_sum / num_users if num_users > 0 else 0
micro_recall = total_hits / total_relevant if total_relevant > 0 else 0

print("Users evaluated:", num_users)
print("Hits:", total_hits)
print("Possible hits:", total_relevant)
print(f"Precision@100: {avg_precision:.4f}")
print(f"Recall@100: {avg_recall:.4f}")
print(f"NDCG@100: {avg_ndcg:.4f}")
print(f"Micro Recall@100: {micro_recall:.4f}")

Users evaluated: 6015
Hits: 52759
Possible hits: 114951
Precision@100: 0.0877
Recall@100: 0.5190
NDCG@100: 0.3737
Micro Recall@100: 0.4590
